# 0.5.4 - XGBoost Without the Radon Map

Ablation: how much does the model rely on the FOPH radon map? The map was computed from the same measurement database, including the test buildings, so it may leak information about them (Kropat et al., 2014).

The model is retrained without the two radon map predictors (exceedance probability and confidence index), on the same building-level subsets and with the hyperparameters tuned in 0.5. They were tuned with the map, so the scores without it are, if anything, slightly pessimistic. The decision thresholds are recalibrated on the validation subset as in 0.5.1: the map is no longer a predictor, but it is still the reference the thresholds are matched to.

## Setup

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit

from evaluation import (
    RADON_MAP_COLUMN,
    bootstrap_ci,
    by_building,
    load_inhabited,
    load_radon_map,
    load_test_egid,
    load_test_set,
    map_matched_threshold,
    radon_map_comparison,
    regression_scores,
    test_scores,
)

ROOT = '../../'
DIR = '../../models/'
RANDOM_SEED = 42
RADON_MAP_FEATURES = [RADON_MAP_COLUMN, 'Rdn Map - Index de confiance dépassement valeur réf']
RADON_MAP_CUTOFFS = [0.01, 0.10, 0.20]  # FOPH legend class boundaries

## Data

Same training/validation split as 0.5.

In [ ]:
X_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-train.csv', index_col=0).replace([np.inf, -np.inf], np.nan)
y_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-train.csv', index_col=0).squeeze()
egid_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-egid-train.csv', index_col=0)['EGID']
X_test, y_test = load_test_set(ROOT)
egid_test = load_test_egid(ROOT)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, val_idx = next(splitter.split(X_train, y_train, groups=egid_train.loc[X_train.index]))
X_fit, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
y_fit, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    full_preprocessor = pickle.load(file)
with open(f'{DIR}0.5-xgboost-best-params.json') as file:
    best_params = json.load(file)

# 0.2 pipeline without the radon map columns
preprocessor = clone(full_preprocessor).set_params(transformers=[
    (name, transformer, [column for column in columns if column not in RADON_MAP_FEATURES])
    for name, transformer, columns in full_preprocessor.transformers
])

X_fit.shape, X_val.shape, X_test.shape

## Models

As in 0.5: a model trained on the training subset gives the validation predictions used for the thresholds, and the final model is retrained on training and validation.

In [ ]:
def fit(X, y):
    """Preprocessor (without the radon map) and XGBoost model fitted on X, y."""
    fitted = clone(preprocessor).fit(X)
    model = xgb.XGBRegressor(**best_params, objective='reg:squarederror', n_jobs=-1, random_state=RANDOM_SEED)
    model.fit(fitted.transform(X), np.log1p(y))
    return lambda X: np.expm1(model.predict(fitted.transform(X)))


predict_val = fit(X_fit, y_fit)
predict_test = fit(pd.concat([X_fit, X_val]), pd.concat([y_fit, y_val]))

full_model = xgb.XGBRegressor()
full_model.load_model(f'{DIR}0.5-xgboost-log-model.json')

predictions = pd.DataFrame({
    'With radon map': np.expm1(full_model.predict(full_preprocessor.transform(X_test))),
    'Without radon map': predict_test(X_test),
}, index=X_test.index)
val = {
    'With radon map': pd.read_csv(f'{DIR}0.5-xgboost-val-predictions.csv', index_col=0),
    'Without radon map': pd.DataFrame({'observed': y_val, 'predicted': predict_val(X_val)}),
}
for frame in val.values():
    frame['radon_map'] = load_radon_map(ROOT, frame.index)

## Test subset

Decision threshold of each model matched to the recall of the radon map > 1 % on its validation predictions (0.5.1).

In [ ]:
with open(f'{DIR}0.5-decision-threshold.json') as file:
    RADON_REFERENCE_LEVEL = json.load(file)['reference_level']

thresholds = {name: map_matched_threshold(frame, RADON_MAP_CUTOFFS[0], RADON_REFERENCE_LEVEL)[1] for name, frame in val.items()}
print({name: round(threshold) for name, threshold in thresholds.items()})

pd.concat({
    name: bootstrap_ci(lambda t, p: test_scores(t, p, thresholds[name], RADON_REFERENCE_LEVEL), y_test, predictions[name], egid_test)
    for name in predictions
}, axis=1)

Difference (with − without), on the same bootstrap samples of buildings.

In [ ]:
def difference(t, p):
    with_map, without_map = (test_scores(t, p[:, i], thresholds[name], RADON_REFERENCE_LEVEL) for i, name in enumerate(predictions))
    return {metric: with_map[metric] - without_map[metric] for metric in ('R2 (log)', 'Average precision')}


bootstrap_ci(difference, y_test, predictions.to_numpy(), egid_test)

## Comparison with the FOPH radon map

As in 0.5.3: test buildings, inhabited rooms only, highest observed and predicted concentration per building, model thresholds set on validation buildings to reach the recall of the map.

In [ ]:
egid_val = egid_train.loc[val['Without radon map'].index]
inhabited_test = load_inhabited(ROOT, X_test.index)
comparisons = {}
for name in predictions:
    test = pd.DataFrame({'observed': y_test, 'predicted': predictions[name], 'radon_map': X_test[RADON_MAP_COLUMN]})[inhabited_test]
    val_inhabited = val[name][load_inhabited(ROOT, val[name].index)]
    comparisons[name] = radon_map_comparison(by_building(test, egid_test.loc[test.index]),
                                             by_building(val_inhabited, egid_val.loc[val_inhabited.index]),
                                             RADON_MAP_CUTOFFS, RADON_REFERENCE_LEVEL)

comparison = comparisons['With radon map'].query("Method == 'Radon map'")
for name, frame in comparisons.items():
    comparison = pd.concat([comparison, frame.query("Method == 'Model'").assign(Method=f'Model, {name.lower()}')])
comparison.sort_values('Map cut-off', kind='stable').reset_index(drop=True)